# Naive Bayes Model: Employee Attrition Prediction

โมเดล **Gaussian Naive Bayes** สำหรับทำนายการลาออกของพนักงาน (`attrition`: 0 = No Attrition, 1 = Attrition)

ออกแบบให้เทียบกับ KNN (`03_knn_model.ipynb`) ได้โดยตรง:
* ใช้ข้อมูลเดียวกัน (`data_preprocess.csv`)
* Train-test split แบบเดียวกัน (80/20, stratified, `random_state=42`) → test set ชุดเดียวกัน
* 5-Fold Stratified Cross-Validation และใช้ **Macro F1** เป็นตัววัดหลัก เพราะข้อมูลไม่สมดุล

**Experiments**
1. Feature set: ทุก feature (baseline) / 11 features เดียวกับ KNN / 9 features จาก EDA
2. วิธีจัดการ class imbalance: ไม่จัดการ (baseline) / ปรับ prior ให้สมดุล / SMOTE
3. ปรับค่า `var_smoothing` ด้วย Grid Search ในแต่ละ configuration (เลือกด้วย Macro F1 และรายงาน F1 ของแต่ละคลาส)

## IMPORT

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn import metrics
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    make_scorer,
    balanced_accuracy_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42
RESULTS_DIR = '../results/naive_bayes'
os.makedirs(RESULTS_DIR, exist_ok=True)

## LOAD DATA

In [2]:
NB_data = pd.read_csv('../data/data_preprocess.csv')

# คอลัมน์ one-hot (True/False) แปลงเป็น 0/1 เพื่อให้เป็นตัวเลขทั้งหมด
bool_cols = NB_data.select_dtypes(include='bool').columns
NB_data[bool_cols] = NB_data[bool_cols].astype(int)

print("Data shape:", NB_data.shape)
print("\nFirst 5 rows:")
print(NB_data.head(5))

Data shape: (43439, 34)

First 5 rows:
   age  job_level  tenure_years  salary_usd  salary_vs_market_pct  \
0   26          0           5.5       70000                   5.2   
1   35          0           0.5       64000                 -15.5   
2   23          2           2.0      142000                 -15.9   
3   20          4           2.2      147000                   2.6   
4   40          2           6.0      141000                  11.3   

   rto_mandate  days_in_office_required  commute_minutes  prefers_remote  \
0            0                        4               44               1   
1            0                        5               48               1   
2            1                        3               35               1   
3            0                        5               16               0   
4            1                        4               55               1   

   flexibility_importance  ...  department_Data  department_Engineering  \
0             

## CHECK DATA

In [3]:
print("DATA OVERVIEW")
print(f"Total rows: {len(NB_data)}")
print(f"Total columns: {len(NB_data.columns)}")
print(f"Missing values: {NB_data.isnull().sum().sum()}")
print(f"Non-numeric columns: {list(NB_data.select_dtypes(exclude='number').columns)}")

DATA OVERVIEW
Total rows: 43439
Total columns: 34
Missing values: 0
Non-numeric columns: []


## CHECK TARGET IMBALANCE

In [4]:
print("TARGET CLASS DISTRIBUTION")
print("\nClass counts:")
print(NB_data['attrition'].value_counts())
print("\nClass proportion (%):")
print(NB_data['attrition'].value_counts(normalize=True) * 100)

TARGET CLASS DISTRIBUTION

Class counts:
attrition
0    35508
1     7931
Name: count, dtype: int64

Class proportion (%):
attrition
0    81.742213
1    18.257787
Name: proportion, dtype: float64


## SELECT FEATURE SETS

| Feature Set | จำนวน | ที่มา |
| :--- | :--- | :--- |
| **All features** (baseline) | 33 | ทุกคอลัมน์ยกเว้น `attrition` |
| **KNN features** | 11 | ชุดเดียวกับที่ใช้ใน `03_knn_model.ipynb` |
| **EDA features** | 9 | ตัวแปรที่ `02_data_visualization.ipynb` พบว่าสัมพันธ์กับการลาออก |

**EDA features** มาจาก insight ใน EDA:
* `burnout_score`: คนลาออก burnout สูงกว่า
* `engagement_score`: คนลาออก engagement ต่ำกว่า
* `work_arrangement_onsite`, `work_arrangement_remote`: คนที่ทำงาน onsite ลาออกมากที่สุด
* `tenure_years`: คนอายุงานน้อยลาออกมากกว่า
* `salary_vs_market_pct`: คนที่เงินเดือนต่ำกว่าตลาดลาออกมากกว่า
* `after_hours_work`: คนที่ทำงานนอกเวลาลาออกมากกว่า
* `commute_minutes`: ยิ่งเดินทางนาน ยิ่งลาออกมาก
* `manager_relationship_score`: correlation เป็นลบกับการลาออก (กราฟ correlation)
* ไม่ใส่ `age` เพราะ EDA พบว่าการกระจายของสองกลุ่มคล้ายกัน

In [5]:
KNN_FEATURES = [
    'work_arrangement_onsite',
    'days_in_office_required',
    'rto_mandate',
    'burnout_score',
    'commute_minutes',
    'prefers_remote',
    'recent_layoff_round',
    'after_hours_work',
    'job_level',
    'salary_usd',
    'weekly_hours',
]

ALL_FEATURES = [c for c in NB_data.columns if c != 'attrition']

EDA_FEATURES = [
    'burnout_score',
    'engagement_score',
    'work_arrangement_onsite',
    'work_arrangement_remote',
    'tenure_years',
    'salary_vs_market_pct',
    'after_hours_work',
    'commute_minutes',
    'manager_relationship_score',
]

FEATURE_SETS = {
    'All features': ALL_FEATURES,
    'KNN features': KNN_FEATURES,
    'EDA features': EDA_FEATURES,
}

X = NB_data[ALL_FEATURES]
y = NB_data['attrition']

for name, cols in FEATURE_SETS.items():
    print(f"{name}: {len(cols)} features")
print("\nAll features:", ALL_FEATURES)

All features: 33 features
KNN features: 11 features
EDA features: 9 features

All features: ['age', 'job_level', 'tenure_years', 'salary_usd', 'salary_vs_market_pct', 'rto_mandate', 'days_in_office_required', 'commute_minutes', 'prefers_remote', 'flexibility_importance', 'ai_tools_adoption', 'weekly_hours', 'after_hours_work', 'burnout_score', 'manager_relationship_score', 'manager_1on1_per_month', 'engagement_score', 'promotions_last_2y', 'career_growth_score', 'recent_layoff_round', 'team_size', 'gender_Male', 'gender_Other', 'department_Data', 'department_Engineering', 'department_Finance', 'department_HR', 'department_Marketing', 'department_Operations', 'department_Product', 'department_Sales', 'work_arrangement_onsite', 'work_arrangement_remote']


## TRAIN-TEST SPLIT

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("TRAIN-TEST SPLIT (Stratified)")
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining set class distribution:\n{y_train.value_counts()}")
print(f"\nTest set class distribution:\n{y_test.value_counts()}")

TRAIN-TEST SPLIT (Stratified)
Training set size: 34751
Test set size: 8688

Training set class distribution:
attrition
0    28406
1     6345
Name: count, dtype: int64

Test set class distribution:
attrition
0    7102
1    1586
Name: count, dtype: int64


## DEFINE PIPELINES

| Strategy | Pipeline | หมายเหตุ |
| :--- | :--- | :--- |
| Baseline | Scaler → GaussianNB | ใช้ prior ตามสัดส่วนจริง (~82/18) |
| Balanced Prior | Scaler → GaussianNB(priors=[0.5, 0.5]) | วิธีเฉพาะของ Naive Bayes: ให้ทั้งสองคลาสมีน้ำหนักเท่ากันโดยไม่สร้างข้อมูลใหม่ |
| SMOTE | Scaler → SMOTE → GaussianNB | สังเคราะห์ตัวอย่างคลาส Attrition เพิ่มใน training fold เท่านั้น |

`StandardScaler` ใส่ไว้ทุก pipeline เพราะ SMOTE ใช้ระยะทาง และ `var_smoothing` ของ GaussianNB คิดเป็นสัดส่วนของ variance สูงสุด การสเกลจึงทำให้ smoothing มีผลกับทุก feature เท่า ๆ กัน

In [7]:
def build_pipeline(strategy):
    if strategy == 'Baseline':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('nb', GaussianNB())
        ])
    if strategy == 'Balanced Prior':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('nb', GaussianNB(priors=[0.5, 0.5]))
        ])
    if strategy == 'SMOTE':
        return ImbPipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('nb', GaussianNB())
        ])
    raise ValueError(strategy)

STRATEGIES = ['Baseline', 'Balanced Prior', 'SMOTE']

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

var_smoothing_values = np.logspace(-12, 2, 15)

## EXPERIMENT: FEATURE SET × IMBALANCE STRATEGY × VAR_SMOOTHING

ทุก configuration ใช้ Grid Search หา `var_smoothing` ที่ดีที่สุดด้วย 5-Fold Stratified CV โดย**เลือกจาก Macro F1** และบันทึก **F1 ของแต่ละคลาส** (No Attrition / Attrition) ไว้ประกอบการประเมิน เพราะ Macro F1 ที่สูงอาจซ่อน F1 ของคลาส Attrition ที่ต่ำไว้

* `var_smoothing` น้อยมาก (เช่น 1e-12) ≈ ไม่มี smoothing คะแนนมักเท่ากัน Grid Search จึงเลือกค่าแรกสุด
* `var_smoothing` สูงมาก (≥ 1) ทำให้ variance ของทุก feature ใกล้เคียงกัน โมเดลจะเข้าใกล้ linear classifier ถ้าค่าที่ดีที่สุดชนขอบบน แปลว่าข้อมูลเหมาะกับการตัดสินใจแบบ linear มากกว่า

In [ ]:
# Macro F1 ใช้เลือกโมเดล (refit) ส่วน F1 รายคลาสเก็บไว้ประเมินผล
SCORING = {
    'macro_f1': 'f1_macro',
    'f1_no_attrition': make_scorer(f1_score, pos_label=0),
    'f1_attrition': make_scorer(f1_score, pos_label=1),
}

experiment_results = []
smoothing_results = []
search_objects = {}

for fs_name, fs_cols in FEATURE_SETS.items():
    for strategy in STRATEGIES:
        search = GridSearchCV(
            build_pipeline(strategy),
            param_grid={'nb__var_smoothing': var_smoothing_values},
            scoring=SCORING,
            refit='macro_f1',
            cv=cv,
            n_jobs=-1
        )
        search.fit(X_train[fs_cols], y_train)
        search_objects[(fs_name, strategy)] = search

        res = search.cv_results_
        best_idx = search.best_index_
        experiment_results.append({
            'Feature_Set': fs_name,
            'Strategy': strategy,
            'N_Features': len(fs_cols),
            'Best_Var_Smoothing': search.best_params_['nb__var_smoothing'],
            'CV_Macro_F1_Mean': res['mean_test_macro_f1'][best_idx],
            'CV_Macro_F1_Std': res['std_test_macro_f1'][best_idx],
            'CV_F1_No_Attrition_Mean': res['mean_test_f1_no_attrition'][best_idx],
            'CV_F1_No_Attrition_Std': res['std_test_f1_no_attrition'][best_idx],
            'CV_F1_Attrition_Mean': res['mean_test_f1_attrition'][best_idx],
            'CV_F1_Attrition_Std': res['std_test_f1_attrition'][best_idx],
        })

        for i, vs in enumerate(var_smoothing_values):
            smoothing_results.append({
                'Feature_Set': fs_name,
                'Strategy': strategy,
                'Var_Smoothing': vs,
                'CV_Macro_F1': res['mean_test_macro_f1'][i],
                'CV_F1_No_Attrition': res['mean_test_f1_no_attrition'][i],
                'CV_F1_Attrition': res['mean_test_f1_attrition'][i],
            })

        best_vs = search.best_params_['nb__var_smoothing']
        print(f"{fs_name:13s} | {strategy:14s} | var_smoothing={best_vs:.0e} "
              f"| Macro F1={search.best_score_:.4f} "
              f"| F1 No Attr={res['mean_test_f1_no_attrition'][best_idx]:.4f} "
              f"| F1 Attr={res['mean_test_f1_attrition'][best_idx]:.4f}")

experiment_results_df = (
    pd.DataFrame(experiment_results)
    .sort_values('CV_Macro_F1_Mean', ascending=False)
    .reset_index(drop=True)
)
smoothing_results_df = pd.DataFrame(smoothing_results)

All features  | Baseline       | var_smoothing=1e-02 | Macro F1=0.6989 | F1 No Attr=0.8939 | F1 Attr=0.5040
All features  | Balanced Prior | var_smoothing=1e+02 | Macro F1=0.6390 | F1 No Attr=0.7908 | F1 Attr=0.4871  <- at upper edge of the grid


## COMPARE ALL EXPERIMENTS

In [ ]:
print("COMPARISON: ALL EXPERIMENTS (5-Fold CV Results)\n")
print(experiment_results_df.to_string(index=False))

# ตารางเทียบ Feature Set × Strategy (CV Macro F1)
feature_set_comparison = experiment_results_df.pivot(
    index='Feature_Set', columns='Strategy', values='CV_Macro_F1_Mean'
).reindex(index=list(FEATURE_SETS), columns=STRATEGIES)
print("\nCV Macro F1: Feature Set × Strategy\n")
print(feature_set_comparison.round(4).to_string())

# ตารางเทียบ Feature Set × Strategy (CV F1 ของคลาส Attrition)
attrition_f1_comparison = experiment_results_df.pivot(
    index='Feature_Set', columns='Strategy', values='CV_F1_Attrition_Mean'
).reindex(index=list(FEATURE_SETS), columns=STRATEGIES)
print("\nCV F1 (Attrition): Feature Set × Strategy\n")
print(attrition_f1_comparison.round(4).to_string())

In [ ]:
# F1 รายคลาสของแต่ละ configuration
per_class_plot = experiment_results_df.melt(
    id_vars=['Feature_Set', 'Strategy'],
    value_vars=['CV_F1_No_Attrition_Mean', 'CV_F1_Attrition_Mean', 'CV_Macro_F1_Mean'],
    var_name='Metric', value_name='CV_Score'
)
per_class_plot['Metric'] = per_class_plot['Metric'].map({
    'CV_F1_No_Attrition_Mean': 'F1 No Attrition',
    'CV_F1_Attrition_Mean': 'F1 Attrition',
    'CV_Macro_F1_Mean': 'Macro F1',
})

g = sns.catplot(
    data=per_class_plot, kind='bar',
    x='Strategy', y='CV_Score', hue='Metric', col='Feature_Set',
    order=STRATEGIES, height=4, aspect=1.1
)
g.set(ylim=(0, 1))
g.fig.suptitle("CV F1 by Class (5-Fold Stratified CV)", y=1.03)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=experiment_results_df,
    x='Strategy',
    y='CV_Macro_F1_Mean',
    hue='Feature_Set',
    order=STRATEGIES
)
plt.ylim(
    experiment_results_df['CV_Macro_F1_Mean'].min() - 0.05,
    experiment_results_df['CV_Macro_F1_Mean'].max() + 0.03
)
plt.ylabel("CV Macro F1")
plt.title("Naive Bayes: Imbalance Strategy × Feature Set (5-Fold CV)")
plt.legend(title='Feature Set')
plt.tight_layout()
plt.show()

In [ ]:
g = sns.relplot(
    data=smoothing_results_df,
    x='Var_Smoothing',
    y='CV_Macro_F1',
    hue='Strategy',
    col='Feature_Set',
    kind='line',
    marker='o',
    height=4,
    aspect=1.3
)
g.set(xscale='log')
g.fig.suptitle("var_smoothing Selection (5-Fold Stratified CV)", y=1.03)
plt.show()

## SELECT BEST MODEL

In [ ]:
best_row = experiment_results_df.iloc[0]
best_feature_set = best_row['Feature_Set']
best_strategy = best_row['Strategy']
best_features = FEATURE_SETS[best_feature_set]
best_var_smoothing = best_row['Best_Var_Smoothing']

final_pipeline = search_objects[(best_feature_set, best_strategy)].best_estimator_

print("SELECT BEST MODEL")
print(f"\nSelected: Gaussian NB | {best_strategy} | {best_feature_set}")
print(f"  Features: {len(best_features)}")
print(f"  var_smoothing: {best_var_smoothing:.0e}")
print(f"  CV Macro F1:          {best_row['CV_Macro_F1_Mean']:.4f} (±{best_row['CV_Macro_F1_Std']:.4f})")
print(f"  CV F1 No Attrition:   {best_row['CV_F1_No_Attrition_Mean']:.4f} (±{best_row['CV_F1_No_Attrition_Std']:.4f})")
print(f"  CV F1 Attrition:      {best_row['CV_F1_Attrition_Mean']:.4f} (±{best_row['CV_F1_Attrition_Std']:.4f})")

## FINAL TEST SET EVALUATION

In [ ]:
final_pipeline.fit(X_train[best_features], y_train)

y_pred = final_pipeline.predict(X_test[best_features])
y_prob = final_pipeline.predict_proba(X_test[best_features])[:, 1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
balanced_acc = balanced_accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"\nTest Set Metrics:")
print(f"  Accuracy:           {accuracy:.4f}")
print(f"  Precision:          {precision:.4f}")
print(f"  Recall:             {recall:.4f}")
print(f"  F1-score:           {f1:.4f}")
print(f"  Macro F1:           {macro_f1:.4f}")
print(f"  Balanced Accuracy:  {balanced_acc:.4f}")
print(f"  ROC-AUC:            {roc_auc:.4f}")

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No Attrition', 'Attrition']))

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix=conf_matrix,
    display_labels=['No Attrition', 'Attrition']
)
cm_display.plot(cmap='Blues')
plt.title(f'Confusion Matrix - Naive Bayes ({best_strategy}) (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'Naive Bayes (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Naive Bayes (Test Set)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## FEATURE INSIGHT

GaussianNB เก็บค่าเฉลี่ยของแต่ละ feature แยกตามคลาส (`theta_`) ข้อมูลผ่าน `StandardScaler` แล้ว ผลต่างของค่าเฉลี่ยระหว่างคลาสจึงอยู่ในหน่วย standard deviation และใช้เทียบกันได้ระหว่าง feature

* ค่า **บวก** = กลุ่มที่ลาออกมีค่า feature นี้สูงกว่า
* ค่า **ลบ** = กลุ่มที่ลาออกมีค่า feature นี้ต่ำกว่า
* ขนาดยิ่งมาก = feature ช่วยแยกสองคลาสได้มาก

In [ ]:
nb_model = final_pipeline.named_steps['nb']

feature_insight = pd.DataFrame({
    'Feature': best_features,
    'Mean_No_Attrition': nb_model.theta_[0],
    'Mean_Attrition': nb_model.theta_[1],
})
feature_insight['Mean_Difference'] = (
    feature_insight['Mean_Attrition'] - feature_insight['Mean_No_Attrition']
)
feature_insight = (
    feature_insight
    .reindex(feature_insight['Mean_Difference'].abs().sort_values(ascending=False).index)
    .reset_index(drop=True)
)

print(feature_insight.to_string(index=False))

plot_df = feature_insight.head(15).iloc[::-1]
plt.figure(figsize=(9, max(4, len(plot_df) * 0.4)))
plt.barh(
    plot_df['Feature'],
    plot_df['Mean_Difference'],
    color=['red' if v > 0 else 'blue' for v in plot_df['Mean_Difference']]
)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel("Mean Difference (Attrition − No Attrition, in std units)")
plt.title("Naive Bayes: Class Mean Difference by Feature")
plt.tight_layout()
plt.show()

## COMPARE WITH KNN

In [ ]:
nb_metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score',
               'Macro F1', 'Balanced Accuracy', 'ROC-AUC'],
    'Naive Bayes': [accuracy, precision, recall, f1,
                    macro_f1, balanced_acc, roc_auc]
})

knn_path = '../results/knn/knn_evaluation.csv'
if os.path.exists(knn_path):
    knn_metrics = pd.read_csv(knn_path).rename(columns={'Score': 'KNN'})
    model_comparison = knn_metrics.merge(nb_metrics, on='Metric')
    print(model_comparison.to_string(index=False))

    model_comparison.set_index('Metric').plot(kind='bar', figsize=(10, 5))
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha='right')
    plt.title("KNN vs Naive Bayes (Test Set)")
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    model_comparison = None
    print(f"KNN results not found at {knn_path} — skipping comparison")

## EXPORT RESULTS FOR POWER BI

In [ ]:
nb_metrics.rename(columns={'Naive Bayes': 'Score'}).to_csv(
    f'{RESULTS_DIR}/nb_evaluation.csv', index=False)

experiment_results_df.to_csv(
    f'{RESULTS_DIR}/nb_experiments_comparison.csv', index=False)

feature_set_comparison.reset_index().to_csv(
    f'{RESULTS_DIR}/nb_feature_set_comparison.csv', index=False)

attrition_f1_comparison.reset_index().to_csv(
    f'{RESULTS_DIR}/nb_feature_set_comparison_f1_attrition.csv', index=False)

smoothing_results_df.to_csv(
    f'{RESULTS_DIR}/nb_var_smoothing_selection.csv', index=False)

pd.DataFrame({
    'Actual': ['No Attrition', 'No Attrition', 'Attrition', 'Attrition'],
    'Predicted': ['No Attrition', 'Attrition', 'No Attrition', 'Attrition'],
    'Count': [conf_matrix[0, 0], conf_matrix[0, 1],
              conf_matrix[1, 0], conf_matrix[1, 1]]
}).to_csv(f'{RESULTS_DIR}/nb_confusion_matrix.csv', index=False)

report = classification_report(y_test, y_pred, output_dict=True)
pd.DataFrame({
    'Class': ['No Attrition', 'Attrition'],
    'Precision': [report['0']['precision'], report['1']['precision']],
    'Recall': [report['0']['recall'], report['1']['recall']],
    'F1-score': [report['0']['f1-score'], report['1']['f1-score']],
    'Support': [int(report['0']['support']), int(report['1']['support'])]
}).to_csv(f'{RESULTS_DIR}/nb_performance.csv', index=False)

pd.DataFrame({
    'Model_Type': [best_strategy],
    'Feature_Set': [best_feature_set],
    'N_Features': [len(best_features)],
    'Best_Var_Smoothing': [best_var_smoothing],
    'CV_Macro_F1': [best_row['CV_Macro_F1_Mean']],
    'CV_F1_No_Attrition': [best_row['CV_F1_No_Attrition_Mean']],
    'CV_F1_Attrition': [best_row['CV_F1_Attrition_Mean']],
    'Test_Accuracy': [round(accuracy * 100, 2)],
    'Test_Macro_F1': [round(macro_f1 * 100, 2)],
    'Test_F1_No_Attrition': [round(report['0']['f1-score'] * 100, 2)],
    'Test_F1_Attrition': [round(report['1']['f1-score'] * 100, 2)],
    'Test_ROC_AUC': [round(roc_auc * 100, 2)]
}).to_csv(f'{RESULTS_DIR}/nb_summary.csv', index=False)

feature_insight.to_csv(f'{RESULTS_DIR}/nb_feature_insight.csv', index=False)

if model_comparison is not None:
    model_comparison.to_csv(f'{RESULTS_DIR}/model_comparison.csv', index=False)

print("Exported files:")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(" -", f)

## NOTES

* **สมมติฐานของ Naive Bayes:** ถือว่าทุก feature เป็นอิสระต่อกันเมื่อรู้คลาส แต่ในข้อมูลนี้มี feature ที่สัมพันธ์กันเอง เช่น `rto_mandate`, `days_in_office_required`, `work_arrangement_onsite` ทำให้ความน่าจะเป็นที่ทำนายได้อาจ "มั่นใจเกินจริง" ลำดับการจัดอันดับ (ROC-AUC) มักยังใช้ได้ดีกว่าตัวความน่าจะเป็นเอง
* **GaussianNB กับตัวแปร 0/1:** Gaussian NB สมมติให้แต่ละ feature กระจายแบบปกติ ซึ่งไม่ตรงกับตัวแปร binary/one-hot แต่ในทางปฏิบัติยังทำงานได้ ถ้าอยากต่อยอด อาจลองแยกตัวแปรต่อเนื่องใช้ GaussianNB และตัวแปร binary ใช้ BernoulliNB แล้วรวมผล
* **Balanced Prior vs SMOTE:** กับ Naive Bayes การปรับ prior ให้สมดุลมีผลคล้าย SMOTE เพราะทั้งสองวิธีเลื่อนจุดตัดสินใจไปทางคลาส Attrition แต่การปรับ prior ไม่ต้องสร้างข้อมูลสังเคราะห์ จึงเร็วกว่าและตีความง่ายกว่า